# Три ДЗ: проверка и запуск проекта в Google Colab

Запускайте ячейки сверху вниз кнопкой ▶. Достаточно **CPU**, GPU не нужен.

1. Загрузите исправленный ZIP проекта в ячейке 1.
2. Дождитесь установки и зелёных проверок.
3. Выполните технический пример на синтетических данных.
4. Для проверки качества на своём датасете включите `RUN_FULL = True` и загрузите Kaggle `train.csv` (`date,store,item,sales`).
5. Скачайте ZIP исходников и отдельный ZIP результатов; инструкции для GitHub — в `START_HERE_RU.md`.

Технический пример не подтверждает историческую sMAPE 12.975%. Исходный датасет не включён в архив. Этот ноутбук использует изолированное Poetry-окружение через subprocess: смена kernel и перезапуск после установки не нужны.

## 1. Загрузить исправленный ZIP

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import files

WORK = Path("/content/retail_hw_run")
WORK.mkdir(parents=True, exist_ok=True)
PROJECT = WORK / "time_series_retail_forecasting"

if not (PROJECT / "pyproject.toml").is_file():
    print("Выберите исправленный ZIP проекта, полученный вместе с этим ноутбуком")
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Нужно выбрать ровно один ZIP проекта")
    with zipfile.ZipFile(archives[0]) as archive:
        for info in archive.infolist():
            destination = (WORK / info.filename).resolve()
            if not destination.is_relative_to(PROJECT.resolve()):
                raise ValueError(f"Неожиданная структура ZIP: {info.filename}")
            if (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Символьные ссылки в ZIP не поддерживаются")
        archive.extractall(WORK)
    del uploaded
if not (PROJECT / "pyproject.toml").is_file():
    raise FileNotFoundError("В ZIP должна быть папка time_series_retail_forecasting")
print("Проект:", PROJECT)

## 2. Установить Poetry и зависимости из lock-файла

Первая установка может занять несколько минут. Вывод не должен заканчиваться ошибкой. Исследовательские DL-библиотеки в этот запуск не входят.

In [ ]:
TOOLS = WORK / "poetry_tools"
if not (TOOLS / "bin/python").is_file():
    subprocess.run([sys.executable, "-m", "venv", str(TOOLS)], check=True)
subprocess.run(
    [str(TOOLS / "bin/python"), "-m", "pip", "install", "-q", "poetry==2.2.1", "uv==0.7.13"],
    check=True,
)
POETRY = str(TOOLS / "bin/poetry")
ENV = os.environ.copy()
ENV.pop("VIRTUAL_ENV", None)
ENV.pop("CONDA_PREFIX", None)
ENV["POETRY_VIRTUALENVS_IN_PROJECT"] = "true"
ENV["POETRY_NO_INTERACTION"] = "1"
PYTHON = sys.executable
if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    # Если Colab изменит Python, используем совместимый Python 3.12 отдельно от kernel.
    UV = str(TOOLS / "bin/uv")
    subprocess.run([UV, "python", "install", "3.12"], check=True, env=ENV)
    PYTHON = subprocess.check_output([UV, "python", "find", "3.12"], text=True, env=ENV).strip()


def run(*args):
    subprocess.run([str(a) for a in args], cwd=PROJECT, env=ENV, check=True)


run(POETRY, "env", "use", PYTHON)
run(POETRY, "check", "--lock")
run(POETRY, "install", "--no-interaction")
run(
    POETRY,
    "run",
    "python",
    "-c",
    "import sys, mlforecast, xgboost; print(sys.version); print(mlforecast.__version__, xgboost.__version__)",
)

## 3. Проверить код и тесты

Создаётся только локальный Git-репозиторий для pre-commit. Эта ячейка ничего не отправляет на GitHub. При автоматическом исправлении форматирования проверка повторяется один раз.

In [ ]:
if not (PROJECT / ".git").is_dir():
    run("git", "init", "-b", "main")
run("git", "add", "--all")
run(POETRY, "run", "pre-commit", "install")
check = subprocess.run([POETRY, "run", "pre-commit", "run", "--all-files"], cwd=PROJECT, env=ENV)
if check.returncode:
    run("git", "add", "--all")
    run(POETRY, "run", "pre-commit", "run", "--all-files")
run(POETRY, "run", "pytest", "-q")
run(POETRY, "build")
print("Проверки успешно завершены")

## 4. Полный технический пример

4 синтетических ряда × 240 дней → backtest → сохранение модели → повторная загрузка → 28-дневный прогноз. Метрики этого примера относятся только к синтетическим данным.

In [ ]:
from datetime import UTC, datetime

RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%f")
DEMO_RESULTS = Path("results") / ("demo_" + RUN_ID)
DEMO_MODEL = Path("artifacts") / ("demo_" + RUN_ID)
run(POETRY, "run", "python", "scripts/make_demo_data.py")
run(
    POETRY,
    "run",
    "retail-forecast",
    "evaluate",
    "--data",
    "data/raw/demo.csv",
    "--config",
    "configs/demo.json",
    "--output",
    DEMO_RESULTS,
    "--expected-series",
    "4",
)
run(
    POETRY,
    "run",
    "retail-forecast",
    "train",
    "--data",
    "data/raw/demo.csv",
    "--config",
    "configs/demo.json",
    "--model-dir",
    DEMO_MODEL,
    "--expected-series",
    "4",
)
run(
    POETRY,
    "run",
    "retail-forecast",
    "predict",
    "--model-dir",
    DEMO_MODEL,
    "--output",
    DEMO_RESULTS / "forecast.csv",
)
print((PROJECT / DEMO_RESULTS / "metrics.csv").read_text())
print("Первые строки прогноза:")
print("\n".join((PROJECT / DEMO_RESULTS / "forecast.csv").read_text().splitlines()[:6]))
RUN_FOLDERS = [DEMO_RESULTS, DEMO_MODEL]

## 5. Проверить настоящий датасет (для пересчёта метрик)

Измените `RUN_FULL = False` на `RUN_FULL = True`. Затем запустите ячейку и выберите **train.csv** из [Store Item Demand Forecasting Challenge](https://www.kaggle.com/competitions/demand-forecasting-kernels-only/data).

Ожидается 500 рядов. Для исходного файла — 913 000 строк, 42 000 проверочных прогнозов на модель, 14 000 будущих прогнозов. `test.csv` и `sample_submission.csv` здесь не нужны. Сначала выполняется backtest, затем отдельная модель обучается на всей истории. При данных до 31.12.2017 прогноз охватывает 01–28.01.2018, а не текущие даты.

In [ ]:
RUN_FULL = False  # Поставьте True, чтобы загрузить и обработать настоящий train.csv.
if RUN_FULL:
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.lower().endswith(".csv")]
    if len(candidates) != 1:
        raise ValueError("Выберите один CSV: train.csv с колонкой sales")
    DATA = PROJECT / "data/raw/train.csv"
    DATA.parent.mkdir(parents=True, exist_ok=True)
    DATA.write_bytes(uploaded[candidates[0]])
    del uploaded
    FULL_RESULTS = Path("results") / ("full_" + RUN_ID)
    FULL_MODEL = Path("artifacts") / ("full_" + RUN_ID)
    run(
        POETRY,
        "run",
        "retail-forecast",
        "evaluate",
        "--data",
        DATA,
        "--config",
        "configs/default.json",
        "--output",
        FULL_RESULTS,
        "--expected-series",
        "500",
    )
    run(
        POETRY,
        "run",
        "retail-forecast",
        "train",
        "--data",
        DATA,
        "--config",
        "configs/default.json",
        "--model-dir",
        FULL_MODEL,
        "--expected-series",
        "500",
    )
    run(
        POETRY,
        "run",
        "retail-forecast",
        "predict",
        "--model-dir",
        FULL_MODEL,
        "--output",
        FULL_RESULTS / "forecast.csv",
    )
    print((PROJECT / FULL_RESULTS / "metrics.csv").read_text())
    RUN_FOLDERS.extend([FULL_RESULTS, FULL_MODEL])
else:
    print("Настоящий датасет не запускался; выполнен только технический пример.")

## 6. Скачать исходники и результаты

`retail_source_for_github.zip` — распакуйте и добавьте файлы в репозиторий. ZIP сам по себе не заменяет исходники на GitHub.

`retail_run_results.zip` — прогнозы, метрики, конфигурация и сохранённые модели; храните отдельно. Этот архив не предназначен для коммита. Виртуальное окружение, исходные данные и `.git` не включаются в архивы. Не загружайте чужие сериализованные модели.

In [ ]:
SOURCE_ZIP = WORK / "retail_source_for_github.zip"
RESULTS_ZIP = WORK / "retail_run_results.zip"
run(POETRY, "run", "python", "scripts/export_zip.py", "--output", SOURCE_ZIP)
with zipfile.ZipFile(RESULTS_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in RUN_FOLDERS:
        for path in sorted((PROJECT / folder).rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(PROJECT))
print("Исходники:", SOURCE_ZIP)
print("Результаты:", RESULTS_ZIP)
files.download(str(SOURCE_ZIP))
files.download(str(RESULTS_ZIP))

## 7. Обновить GitHub и сдать ДЗ

Откройте `START_HERE_RU.md` в распакованном проекте. Рекомендуемый путь: GitHub Desktop → clone → новая ветка `homework-fixes` → заменить файлы содержимым исправленной папки → commit → publish branch → Pull Request → дождаться CI → merge.

Ссылки для сдачи после обновления ветки `main`:

- https://github.com/SofiCholoian888/time_series_retail_forecasting/blob/main/model_card.md
- https://github.com/SofiCholoian888/time_series_retail_forecasting/blob/main/Model_BPMN.md
- https://github.com/SofiCholoian888/time_series_retail_forecasting/blob/main/Model_retrain_BPMN.md

ДЗ №1 подтверждается содержимым `src/`, `tests/`, `pyproject.toml`, `poetry.lock`, `.pre-commit-config.yaml`, `.github/workflows/ci.yml` и успешным запуском CI. Для работы в реальном бизнесе ещё нужны новые данные и реализация процесса из ДЗ №3; сохранение модели командой `train` не является автоматическим деплоем.